# 08 — Temporal granularity feasibility for 2018–2023

This is a **descriptive feasibility check** of the frozen, validated 24-month Freddie Mac modeling dataset. It reads existing data and writes no target, feature, model, or drift artifacts. Run it from the repository root or `notebooks/` after the local processed Parquet is available. Do not interpret tables unless every validation and reconciliation check passes.

## 1. Locate the existing modeling artifact

Notebook 06 saved `freddie_modeling_dataset_2015_2023_24m.parquet`; notebook 07 reads that same file before feature engineering. The feature table lacks the horizon audit field, so this analysis uses the validated modeling table. The source year and origination cohort fields were derived from the Freddie loan identifier in `src/freddie_cohorts.py`, rather than from monthly performance records. This cell reads only the needed columns and fails if the file is absent.

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "Freddie_Mac"
INPUT_PATH = DATA_DIR / "freddie_modeling_dataset_2015_2023_24m.parquet"
VALIDATION_PATH = DATA_DIR / "freddie_modeling_dataset_2015_2023_24m_validation.json"
TARGET = "TARGET_COMPOSITE_CREDIT_EVENT"
COLUMNS = ["SOURCE YEAR", "LOAN IDENTIFIER", "ORIGINATION YEAR",
           "ORIGINATION QUARTER", "ORIGINATION COHORT", "HORIZON_MONTHS", TARGET]

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Validated modeling dataset is unavailable: {INPUT_PATH}. "
        "Copy the existing processed artifact into the project; do not rebuild labels for this analysis."
    )
data = pd.read_parquet(INPUT_PATH, columns=COLUMNS)
print("Dataset:", INPUT_PATH)
print("Target:", TARGET)
print("Rows in complete modeling artifact:", len(data))
if VALIDATION_PATH.is_file():
    with VALIDATION_PATH.open(encoding="utf-8") as stream:
        prior_validation = json.load(stream)
    print("Previous pipeline validation:", VALIDATION_PATH)
else:
    prior_validation = None
    print("Previous pipeline validation JSON unavailable; direct checks below remain required.")

Dataset: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_modeling_dataset_2015_2023_24m.parquet
Target: TARGET_COMPOSITE_CREDIT_EVENT
Rows in complete modeling artifact: 437668
Previous pipeline validation: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_modeling_dataset_2015_2023_24m_validation.json


## 2. Audit cohort provenance, keys, horizon, and label before grouping

`ORIGINATION YEAR`, `ORIGINATION QUARTER`, and `ORIGINATION COHORT` are the period fields. Check them against the encoded origination year and quarter in each loan identifier, as well as `SOURCE YEAR`. Record missing and invalid values, including any OOT rows that could be lost by filtering. Report all failures first; do not silently coerce, repair, or drop them. The target must be present, binary, and at exactly 24 months for every included loan.

In [2]:
expected_years = set(range(2018, 2024))
year_numeric = pd.to_numeric(data["SOURCE YEAR"], errors="coerce")
cohort_year = pd.to_numeric(data["ORIGINATION YEAR"], errors="coerce")
quarter = data["ORIGINATION QUARTER"].astype("string")
cohort = data["ORIGINATION COHORT"].astype("string")
loan_id = data["LOAN IDENTIFIER"].astype("string")
id_parts = loan_id.str.extract(r"^[FA](?P<year>\d{2})Q(?P<quarter>[1-4])\d{7}$")
id_year = pd.to_numeric(id_parts["year"], errors="coerce") + 2000
id_quarter = "Q" + id_parts["quarter"]
expected_cohort = cohort_year.astype("Int64").astype("string") + quarter
is_oot = year_numeric.isin(expected_years)
oot = data.loc[is_oot].copy()
label_numeric = pd.to_numeric(data[TARGET], errors="coerce")
horizon_numeric = pd.to_numeric(data["HORIZON_MONTHS"], errors="coerce")

checks = {
    "complete_artifact_rows": len(data),
    "OOT_rows_by_SOURCE_YEAR": int(is_oot.sum()),
    "missing_SOURCE_YEAR_all": int(data["SOURCE YEAR"].isna().sum()),
    "invalid_SOURCE_YEAR_all": int((year_numeric.isna() & data["SOURCE YEAR"].notna()).sum()),
    "OOT_missing_ORIGINATION_YEAR": int(cohort_year[is_oot].isna().sum()),
    "OOT_missing_ORIGINATION_QUARTER": int(quarter[is_oot].isna().sum()),
    "OOT_missing_ORIGINATION_COHORT": int(cohort[is_oot].isna().sum()),
    "OOT_missing_LOAN_IDENTIFIER": int(loan_id[is_oot].isna().sum()),
    "OOT_invalid_identifier_format": int(id_year[is_oot].isna().sum()),
    "OOT_invalid_quarter_value": int((~quarter[is_oot].isin(["Q1", "Q2", "Q3", "Q4"])).sum()),
    "OOT_year_source_mismatch": int((cohort_year[is_oot].ne(year_numeric[is_oot]).fillna(True)).sum()),
    "OOT_year_identifier_mismatch": int((cohort_year[is_oot].ne(id_year[is_oot]).fillna(True)).sum()),
    "OOT_quarter_identifier_mismatch": int((quarter[is_oot].ne(id_quarter[is_oot]).fillna(True)).sum()),
    "OOT_cohort_year_quarter_mismatch": int((cohort[is_oot].ne(expected_cohort[is_oot]).fillna(True)).sum()),
    "OOT_duplicate_SOURCE_YEAR_loan_keys": int(oot.duplicated(["SOURCE YEAR", "LOAN IDENTIFIER"]).sum()),
    "OOT_missing_target": int(data.loc[is_oot, TARGET].isna().sum()),
    "OOT_invalid_binary_target": int((~label_numeric[is_oot].isin([0, 1])).sum()),
    "OOT_missing_horizon": int(data.loc[is_oot, "HORIZON_MONTHS"].isna().sum()),
    "OOT_non_24_month_horizon": int((horizon_numeric[is_oot].ne(24)).sum()),
    "OOT_years_present": sorted(year_numeric[is_oot].astype(int).unique().tolist()),
}
display(pd.Series(checks, name="Result").to_frame())
if prior_validation is not None:
    prior_checks = {key: prior_validation.get(key) for key in
                    ["selected_horizon_months", "modeling_rows", "duplicate_loan_year_keys",
                     "missing_labels", "year_cohort_mismatches", "readback_validation_matches"]}
    display(pd.Series(prior_checks, name="Previous validation").to_frame())

failures = [key for key, value in checks.items()
            if key.startswith("OOT_") and isinstance(value, int) and
            key != "OOT_rows_by_SOURCE_YEAR" and value != 0]
if checks["OOT_years_present"] != sorted(expected_years):
    failures.append("OOT_years_present")
if year_numeric.isna().any():
    failures.append("SOURCE YEAR missing or invalid in complete artifact; cohort allocation needs review")
if prior_validation is not None and (
    prior_validation.get("selected_horizon_months") != 24 or
    prior_validation.get("modeling_rows") != len(data) or
    prior_validation.get("readback_validation_matches") is not True
):
    failures.append("previous validation mismatch")
if failures:
    raise ValueError("Validation failed; no period tables generated: " + ", ".join(failures))
print("All direct OOT contract checks passed; period grouping can proceed.")

,Result
complete_artifact_rows,437668
OOT_rows_by_SOURCE_YEAR,294070
missing_SOURCE_YEAR_all,0
invalid_SOURCE_YEAR_all,0
OOT_missing_ORIGINATION_YEAR,0
OOT_missing_ORIGINATION_QUARTER,0
OOT_missing_ORIGINATION_COHORT,0
OOT_missing_LOAN_IDENTIFIER,0
OOT_invalid_identifier_format,0
OOT_invalid_quarter_value,0


,Previous validation
selected_horizon_months,24
modeling_rows,437668
duplicate_loan_year_keys,0
missing_labels,0
year_cohort_mismatches,0
readback_validation_matches,True


All direct OOT contract checks passed; period grouping can proceed.


## 3. Count annual, half-year, and quarterly outcomes

A half-year combines Q1–Q2 or Q3–Q4 of the *same origination year*. Every output has the same OOT loan universe. Include all expected periods, including zero-count periods, rather than letting a groupby silently omit one. Event rate is positive loans divided by all eligible loans in that period.

In [3]:
working = oot[["ORIGINATION YEAR", "ORIGINATION QUARTER", TARGET]].copy()
working["year"] = working["ORIGINATION YEAR"].astype(int)
working["quarter_number"] = working["ORIGINATION QUARTER"].str[-1].astype(int)
working["half"] = np.where(working["quarter_number"].le(2), "H1", "H2")

def period_counts(fields, labels):
    grouped = working.groupby(fields, observed=True, dropna=False)[TARGET].agg(
        loans="size", positives="sum")
    grouped = grouped.reindex(labels, fill_value=0).reset_index()
    grouped["loans"] = grouped["loans"].astype(int)
    grouped["positives"] = grouped["positives"].astype(int)
    grouped["negatives"] = grouped["loans"] - grouped["positives"]
    grouped["event_rate_percent"] = (100 * grouped["positives"] /
                                      grouped["loans"].replace(0, np.nan))
    return grouped

years = list(range(2018, 2024))
annual = period_counts(["year"], pd.Index(years, name="year"))
half = period_counts(["year", "half"], pd.MultiIndex.from_product(
    [years, ["H1", "H2"]], names=["year", "half"]))
quarterly = period_counts(["year", "quarter_number"], pd.MultiIndex.from_product(
    [years, [1, 2, 3, 4]], names=["year", "quarter_number"]))
annual.insert(0, "period", annual["year"].astype(str))
half.insert(0, "period", half["year"].astype(str) + "-" + half["half"])
quarterly.insert(0, "period", quarterly["year"].astype(str) + "-Q" + quarterly["quarter_number"].astype(str))

def show_counts(name, table):
    print(name)
    display(table[["period", "loans", "positives", "negatives", "event_rate_percent"]].round(3))

show_counts("Annual 2018–2023", annual)
show_counts("Half-year 2018-H1 through 2023-H2", half)
show_counts("Quarterly 2018-Q1 through 2023-Q4", quarterly)

Annual 2018–2023


,period,loans,positives,negatives,event_rate_percent
0,2018,47417,1222,46195,2.577
1,2019,48611,2227,46384,4.581
2,2020,49665,802,48863,1.615
3,2021,49490,451,49039,0.911
4,2022,49386,903,48483,1.828
5,2023,49501,876,48625,1.770


Half-year 2018-H1 through 2023-H2


,period,loans,positives,negatives,event_rate_percent
0,2018-H1,23699,327,23372,1.380
1,2018-H2,23718,895,22823,3.774
2,2019-H1,24326,1106,23220,4.547
3,2019-H2,24285,1121,23164,4.616
4,2020-H1,24841,607,24234,2.444
5,2020-H2,24824,195,24629,0.786
6,2021-H1,24714,191,24523,0.773
7,2021-H2,24776,260,24516,1.049
8,2022-H1,24633,421,24212,1.709
9,2022-H2,24753,482,24271,1.947


Quarterly 2018-Q1 through 2023-Q4


,period,loans,positives,negatives,event_rate_percent
0,2018-Q1,11884,63,11821,0.530
1,2018-Q2,11815,264,11551,2.234
2,2018-Q3,11771,406,11365,3.449
3,2018-Q4,11947,489,11458,4.093
4,2019-Q1,12149,565,11584,4.651
5,2019-Q2,12177,541,11636,4.443
6,2019-Q3,12084,569,11515,4.709
7,2019-Q4,12201,552,11649,4.524
8,2020-Q1,12425,473,11952,3.807
9,2020-Q2,12416,134,12282,1.079


## 4. Reconcile period totals and outcomes

The half-year and quarterly totals must independently equal their annual counterparts for loans, positives, and negatives. Each granularity must also sum to the complete 2018–2023 sample. Stop if a period is empty or any reconciliation fails, and investigate rather than adjusting the counts.

In [4]:
count_columns = ["loans", "positives", "negatives"]
annual_by_year = annual.set_index("year")[count_columns]
reconciliation = pd.DataFrame(index=annual_by_year.index)
for name, table in [("half_year", half), ("quarter", quarterly)]:
    by_year = table.groupby("year")[count_columns].sum().reindex(years)
    for col in count_columns:
        reconciliation[f"{name}_{col}_difference"] = by_year[col] - annual_by_year[col]
reconciliation.index.name = "year"
display(reconciliation)
expected_total = pd.Series({
    "loans": len(oot),
    "positives": int(oot[TARGET].sum()),
    "negatives": int(len(oot) - oot[TARGET].sum()),
})
overall = pd.DataFrame({name: table[count_columns].sum() for name, table in
                        [("annual", annual), ("half_year", half), ("quarter", quarterly)]})
overall["OOT_direct"] = expected_total
print("Overall counts, including positive and negative reconciliation:")
display(overall)
if not reconciliation.eq(0).all().all() or not overall.eq(expected_total, axis=0).all().all():
    raise ValueError("Period counts fail annual or full OOT reconciliation.")
if any(table["loans"].eq(0).any() for table in [annual, half, quarterly]):
    raise ValueError("At least one expected period has no eligible loans; investigate before interpretation.")
print("PASS: every year's half-year/quarter counts reconcile, and all period sets equal OOT totals.")

,half_year_loans_difference,half_year_positives_difference,half_year_negatives_difference,quarter_loans_difference,quarter_positives_difference,quarter_negatives_difference
year,,,,,,
2018,0,0,0,0,0,0
2019,0,0,0,0,0,0
2020,0,0,0,0,0,0
2021,0,0,0,0,0,0
2022,0,0,0,0,0,0
2023,0,0,0,0,0,0


Overall counts, including positive and negative reconciliation:


,annual,half_year,quarter,OOT_direct
loans,294070,294070,294070,294070
positives,6481,6481,6481,6481
negatives,287589,287589,287589,287589


PASS: every year's half-year/quarter counts reconcile, and all period sets equal OOT totals.


## 5. Compare sample and event support

These are descriptive ranges, not acceptance thresholds. Inspect particularly the smallest positive counts, because subdivisions with few events can make PR-AUC and calibration estimates noisy. Large changes in prevalence also shift the PR-AUC baseline and Brier score even when ranking quality is similar.

In [5]:
summary_rows = []
for name, table in [("Annual", annual), ("Half-year", half), ("Quarter", quarterly)]:
    summary_rows.append({
        "granularity": name, "number_of_periods": len(table),
        "min_loans_per_period": int(table["loans"].min()),
        "median_loans_per_period": float(table["loans"].median()),
        "min_positives_per_period": int(table["positives"].min()),
        "median_positives_per_period": float(table["positives"].median()),
        "max_positives_per_period": int(table["positives"].max()),
        "min_event_rate_percent": float(table["event_rate_percent"].min()),
        "max_event_rate_percent": float(table["event_rate_percent"].max()),
    })
comparison = pd.DataFrame(summary_rows)
display(comparison.round(3))
print("Lowest-event quarters:")
display(quarterly.nsmallest(6, "positives")[["period", "loans", "positives", "negatives", "event_rate_percent"]].round(3))
print("Highest- and lowest-prevalence periods at each granularity:")
for name, table in [("Annual", annual), ("Half-year", half), ("Quarter", quarterly)]:
    extremes = pd.concat([table.nsmallest(1, "event_rate_percent"),
                          table.nlargest(1, "event_rate_percent")])
    print(name)
    display(extremes[["period", "loans", "positives", "event_rate_percent"]].round(3))

,granularity,number_of_periods,min_loans_per_period,median_loans_per_period,min_positives_per_period,median_positives_per_period,max_positives_per_period,min_event_rate_percent,max_event_rate_percent
0,Annual,6,47417,49438.0,451,889.5,2227,0.911,4.581
1,Half-year,12,23699,24729.0,191,440.5,1121,0.773,4.616
2,Quarter,24,11771,12372.0,63,224.5,569,0.530,4.709


Lowest-event quarters:


,period,loans,positives,negatives,event_rate_percent
0,2018-Q1,11884,63,11821,0.530
12,2021-Q1,12409,78,12331,0.629
10,2020-Q3,12426,90,12336,0.724
11,2020-Q4,12398,105,12293,0.847
13,2021-Q2,12305,113,12192,0.918
14,2021-Q3,12388,126,12262,1.017


Highest- and lowest-prevalence periods at each granularity:
Annual


,period,loans,positives,event_rate_percent
3,2021,49490,451,0.911
1,2019,48611,2227,4.581


Half-year


,period,loans,positives,event_rate_percent
6,2021-H1,24714,191,0.773
3,2019-H2,24285,1121,4.616


Quarter


,period,loans,positives,event_rate_percent
0,2018-Q1,11884,63,0.530
6,2019-Q3,12084,569,4.709


## Feasibility conclusion

All validation checks passed. The 2018–2023 OOT sample contains 294,070 eligible loans, including 6,481 positive and 287,589 negative composite 24-month outcomes. Annual, half-year, and quarterly counts reconcile exactly for loans and both outcome classes. Origination cohort fields agree with the loan identifiers; no OOT cohort values, targets, or horizons are missing or invalid, and no loan/year keys are duplicated.

Annual cohorts provide six periods, with 451–2,227 positive events per period. Half-year cohorts provide twelve periods, with 191–1,121 events. Quarterly cohorts provide twenty-four periods, but event support ranges from 63 to 569. Quarterly prevalence ranges from 0.530% to 4.709%; 2018-Q1 has only 63 events, while 2021-Q1 has 78.

Quarterly ROC-AUC and PR-AUC estimates may be imprecise in the low-event periods. PR-AUC must be interpreted alongside each period's prevalence. Brier scores can be calculated, but prevalence changes affect comparisons. Calibration curves and ECE require an explicitly chosen binning procedure, event counts per bin, and uncertainty assessment.

**Provisional recommendation:** Consider half-year cohorts as the primary unit for frozen-model reliability reporting, annual cohorts for context, and quarters for descriptive timing and sensitivity analysis. This is a methodological judgment from event support, not a universal minimum-event rule. Six annual observations do not justify a simple drift–performance regression, and twelve half-year observations still require a restrained association analysis.

Before locking the temporal design, investigate the pronounced 2018 quarterly event-rate change against existing eligibility and target-validation summaries. Later methodology must also distinguish population change observable at origination from reliability measured after the 24-month outcome matures. No drift metrics or model performance metrics were calculated in this notebook.